# Dual-system LIBERO experiments (Colab T4)
Thin entry point: all logic lives in `src/single_system/`. Runtime must be **T4 GPU** before running. Rules and pitfalls: `.claude/skills/colab-startup/SKILL.md`.

In [ ]:
# 1. get the repo (set REPO_URL once; private repos need a token in Colab secrets: userdata.get('GH_TOKEN'))
import os, subprocess
REPO_URL = 'https://github.com/<user>/<repo>.git'
REPO = '/content/repo'
sh = lambda c: print(subprocess.run(c, shell=True, capture_output=True, text=True, executable='/bin/bash'))
if os.path.isdir(REPO): sh(f'cd {REPO} && git pull')
else: sh(f'git clone {REPO_URL} {REPO}')
sh(f'pip install -q -e {REPO}')
print(subprocess.run('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. one-time per runtime: lerobot + LIBERO (~5 min, run in background and poll), then EGL libs + renderer check
import subprocess
subprocess.Popen('nohup python -m single_system.colab install > /content/install.log 2>&1 &', shell=True, start_new_session=True)

In [ ]:
# 2b. after the install finished (tail /content/install.log shows 'lerobot-eval' path)
from single_system import colab
colab.extract_egl()
colab.check_renderer()   # must print Tesla T4

In [ ]:
# 3. real-world frames for the visual-OOD probe (needs lerobot installed)
from single_system.real_clips import build_droid_clips
build_droid_clips()

In [ ]:
# 4. queue the experiments (resumable: finished cells are skipped). Edit grids in single_system/experiments.py
import subprocess
from single_system import experiments as E
from single_system.runner import save_queue
cells = E.queue_A() + E.queue_B1() + E.queue_B2() + E.queue_C()
subprocess.run('mkdir -p /content/res', shell=True)
save_queue(cells, '/content/res/queue.json')   # lives in res/ so every sync carries the exact grid
subprocess.Popen('nohup python -m single_system.runner /content/res/queue.json /content/res > /content/queue.out 2>&1 &', shell=True, start_new_session=True)
print(len(cells), 'cells')

In [ ]:
# 5. progress (keep waits short, a long sleep cell can be interrupted and loses its output)
import subprocess, time
time.sleep(120)
print(subprocess.run('cat /content/res/queue.log | tail -n 15', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 6. results so far: success with Wilson 95% CI per condition (partial cells included; filter df.cell_done for finished only)
from single_system import analysis as A
df = A.load_manifest('/content/res')
A.success_table(df)

In [ ]:
# 7. sync: zip everything so far (running cells included) and download through the browser.
# Locally: powershell -File scripts/pull_results.ps1 mirrors the newest zip into data/system1-latest.
from single_system import colab
colab.sync_to_browser('/content/res')